In [1]:
from pathlib import Path
import json

from app.schemas import GenerationProfile, QuestionDepth, GeneratedQA
from app.rag.embedding import EmbeddingManager
from app.rag.vector_store import VectorStore
from app.rag.retriever import Retriever
from app.generation.qa_generator import QAGenerator

/Users/macstudio/Desktop/Development/DomainForge V1/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASE_DIR = Path.cwd()
VECTORSTORE_DIR = BASE_DIR / "data" / "vectorstore"
OUTPUT_PATH = BASE_DIR / "data" / "processed" / "generated_qa.jsonl"

In [3]:
embed_mgr = EmbeddingManager(model_name="BAAI/bge-base-en-v1.5")
vector_store = VectorStore(
    persist_dir=VECTORSTORE_DIR,
    collection_name="domainforge_governance",
    embedding_manager=embed_mgr
)

retriever = Retriever(vector_store=vector_store, top_k=4)
print(f"Vector Store Connection Successful: {vector_store.count()} records found.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 23258.83it/s]


Vector Store Connection Successful: 2347 records found.


In [4]:
seed_query = "What are the technical risk management characteristics and accountability principles in trustworthy AI?"
retrieved_contexts = retriever.retrieve(seed_query)

print(f"Number of Retrieved Contexts: {len(retrieved_contexts)}\n")
for i, ctx in enumerate(retrieved_contexts, 1):
    print(f"[{i}] Score: {ctx.score:.4f} | {ctx.source} (s. {ctx.page}) | Chunk: {ctx.chunk_id}")

Number of Retrieved Contexts: 4

[1] Score: 0.7569 | NIST.AI.600-1.pdf (s. 13) | Chunk: NIST.AI.600-1_p13_c12
[2] Score: 0.7504 | NIST.AI.600-1.pdf (s. 16) | Chunk: NIST.AI.600-1_p16_c11
[3] Score: 0.7479 | NIST.AI.100-1.pdf (s. 17) | Chunk: NIST.AI.100-1_p17_c4
[4] Score: 0.7433 | NIST.AI.100-1.pdf (s. 7) | Chunk: NIST.AI.100-1_p7_c15


In [ ]:
profile = GenerationProfile(
    question_count=1,
    answer_length="detailed",
    depth=QuestionDepth.TECHNICAL,
    temperature=0.2
)

generator = QAGenerator(model_name="deepseek-r1:14b")

print("Batch Q&A generation is starting...")
qa_dataset = generator.generate_batch(retrieved_contexts, profile)
print(f"Production Complete! Total number of Q/A pairs produced: {len(qa_dataset)}")

Batch Q&A generation is starting...
Production Complete! Total number of Q/A pairs produced: 4


In [6]:
for i, qa in enumerate(qa_dataset, 1):
    print(f"\n{'='*70}")
    print(f"Q/A ITEM #{i}")
    print(f"{'='*70}")
    print(f"QUESTION : {qa.question}")
    print(f"ANSWER   : {qa.answer}")
    print(f"SOURCE   : {qa.source} (Page {qa.page})")
    print(f"CHUNK ID : {qa.chunk_id}")
    print(f"METRICS  : Depth={qa.depth} | Retrieval Score={qa.retrieval_score:.4f}")


Q/A ITEM #1
QUESTION : What are the key technical characteristics of Trustworthy AI as outlined in NIST.AI.600-1.pdf on page 13, chunk NIST.AI.600-1_p13_c12?
ANSWER   : The key technical characteristics of Trustworthy AI, as outlined in NIST.AI.600-1.pdf on page 13, chunk NIST.AI.600-1_p13_c12, include being Accountable and Transparent, Explainable and Interpretable, Fair with Harmful Bias Managed, Privacy Enhanced, Safe, and Valid and Reliable. These characteristics ensure that AI systems are not only effective but also trustworthy by addressing accountability, transparency, explainability, fairness, privacy, safety, and reliability.
SOURCE   : NIST.AI.600-1.pdf (Page 13)
CHUNK ID : NIST.AI.600-1_p13_c12
METRICS  : Depth=technical | Retrieval Score=0.7569

Q/A ITEM #2
QUESTION : What are the key technical characteristics of Trustworthy AI as outlined in NIST.AI.600-1.pdf on page 16, chunk NIST.AI.600-1_p16_c11?
ANSWER   : The key technical characteristics of Trustworthy AI, as outlin

In [7]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for qa in qa_dataset:
        f.write(qa.model_dump_json() + "\n")

print(f"The synthetic dataset has been successfully saved: {OUTPUT_PATH}")

The synthetic dataset has been successfully saved: /Users/macstudio/Desktop/Development/DomainForge V1/data/processed/generated_qa.jsonl
